# ARC-Challenge — Base Baseline (Standard RAG, Cosine-Only)

**Purpose:** Provide the `Base` row for ARC in the paper table.  
Replicates Li et al. (2025) setup: Mistral-7B-Instruct-v0.2 (4-bit) + cosine retrieval, no contrastive reranking.  
KB = ARC train+val (1418Q), Test = ARC official test (1172Q).

In [1]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import os, sys, gc, json, torch, faiss, numpy as np
from datetime import datetime
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
import mauve as mauve_lib
from datasets import load_from_disk

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED   = 42
CHOICE_LABELS = ['A', 'B', 'C', 'D']
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Standard RAG params (same as Li et al. 2025 Base)
TOP_K_RETRIEVE = 5    # retrieve 5 docs by cosine
TOP_K_CONTEXT  = 3    # use top-3 sentences as context
MAX_GEN_TOKENS = 25
NUM_BEAMS      = 2

print('Config OK')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Config OK
GPU: Quadro RTX 8000


In [2]:
# ── Cell 2: Load ARC-Challenge ────────────────────────────────────────────────
import pandas as pd

def arc_to_unified(row):
    texts  = list(row['choices']['text'])
    labels = list(row['choices']['label'])
    key    = str(row['answerKey'])
    if key.isdigit():
        ans_idx = int(key) - 1
    else:
        ans_idx = labels.index(key) if key in labels else 0
    ans_idx   = min(ans_idx, len(texts) - 1)
    correct   = texts[ans_idx]
    incorrect = [texts[i] for i in range(len(texts)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{labels[i]}) {texts[i]}' for i in range(len(texts))))
    return pd.Series({
        'question':      formatted_q,
        'question_plain': row['question'],
        'best_answer':   [correct],
        'correct_answers': [correct],
        'incorrect_answers': incorrect,
        'answer_idx':    ans_idx,
        'choices':       texts,
    })

print('Loading ARC-Challenge...')
arc_test_raw = load_from_disk(f'{DATASETS_DIR}/arc_challenge_test').to_pandas()
arc_trn_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_train').to_pandas()
arc_val_raw  = load_from_disk(f'{DATASETS_DIR}/arc_challenge_validation').to_pandas()
arc_kb_raw   = pd.concat([arc_trn_raw, arc_val_raw], ignore_index=True)

arc     = arc_test_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
arc_kb  = arc_kb_raw.apply(arc_to_unified, axis=1).reset_index(drop=True)
del arc_test_raw, arc_trn_raw, arc_val_raw, arc_kb_raw; gc.collect()
print(f'  test={len(arc)}Q | KB={len(arc_kb)}Q (train+val, zero overlap)')

Loading ARC-Challenge...
  test=1172Q | KB=1418Q (train+val, zero overlap)


In [3]:
# ── Cell 3: Load Model (4-bit) ────────────────────────────────────────────────
gc.collect(); torch.cuda.empty_cache()

print('Loading Mistral-7B (4-bit)...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(
    f'{MODELS_DIR}/mistral-7b', quantization_config=bnb, device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(
    f'{MODELS_DIR}/mistral-7b', padding_side='left'
)
tokenizer.pad_token = tokenizer.eos_token

print('Loading MiniLM...')
embed_model = SentenceTransformer(f'{MODELS_DIR}/minilm').to(DEVICE)
print('Models ready!')

Loading Mistral-7B (4-bit)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading MiniLM...
Models ready!


In [4]:
# ── Cell 4: Utilities ────────────────────────────────────────────────────────
import re

INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = ('You are a truthful expert question-answering bot and should '
          'correctly and concisely answer the following question')

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id
        )
    return [
        tokenizer.decode(r[input_len:], skip_special_tokens=True).strip() or 'I have no comment'
        for r in out
    ]

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\n\n','\nContext:','\nFor the question']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def build_faiss_index(dataset):
    embs = embed_model.encode(dataset['question'].tolist(),
                               show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def cosine_retrieve(query, faiss_idx, kb, k=5):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = faiss_idx.search(q_emb, k)
    return [kb.iloc[i] for i in idxs[0] if i < len(kb)]

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, r2s, rls, ecss = [], [], [], []
    for gen, refs in zip(generated, references):
        best = [0, 0, 0]
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best[0] = max(best[0], s['rouge1'].fmeasure)
            best[1] = max(best[1], s['rouge2'].fmeasure)
            best[2] = max(best[2], s['rougeL'].fmeasure)
        r1s.append(best[0]*100); r2s.append(best[1]*100); rls.append(best[2]*100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0])*100)
        except: ecss.append(0.0)
    return np.array(r1s), np.array(r2s), np.array(rls), np.array(ecss)

def compute_mauve(generated, references):
    try:
        refs  = [r[0] if isinstance(r, list) else r for r in references]
        valid = [(g, r) for g, r in zip(generated, refs) if g and r]
        if len(valid) < 10: return 0.0
        gens_v, refs_v = zip(*valid)
        result = mauve_lib.compute_mauve(
            p_text=list(refs_v), q_text=list(gens_v),
            device_id=0, max_text_length=256, verbose=False,
            featurize_model_name='gpt2'
        )
        return result.mauve * 100
    except Exception as e:
        print(f'  MAUVE error: {e}'); return 0.0

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0]
        ans_idx = int(row['answer_idx'])
        label   = CHOICE_LABELS[ans_idx]
        if label in gen[:5] or correct_text.lower() in gen.lower():
            correct += 1
    return correct / len(generated) * 100

print('Utilities ready ✓')

Utilities ready ✓


In [5]:
# ── Cell 5: RUN — ARC Base (Standard RAG, cosine-only) ───────────────────────
# This replicates the 'Base' setup from Li et al. (2025):
# retrieve top-k docs by cosine similarity, concatenate as context, generate.
# NO contrastive reranking, NO priors, NO KB labels used.

import time

print('Building FAISS index on ARC KB...')
faiss_idx = build_faiss_index(arc_kb)

generated, references = [], []
t0 = time.time()

print(f'\nRunning ARC Base (Standard RAG) on {len(arc)} questions...')
for idx, row in tqdm(arc.iterrows(), total=len(arc), desc='ARC-Base'):
    query       = row['question']
    best_answer = row['best_answer']

    # Standard cosine retrieval — no labels, no priors
    retrieved = cosine_retrieve(query, faiss_idx, arc_kb, k=TOP_K_RETRIEVE)

    if retrieved:
        # Collect top sentences by concatenating retrieved question texts
        sentences = []
        for doc in retrieved:
            for sent in re.split(r'(?<=[.!?])\s+', doc['question']):
                if len(sent.strip()) > 8:
                    sentences.append(sent.strip())
        context = ' '.join(sentences[:TOP_K_CONTEXT * 2])[:600]
        prompt = (
            f'{INST_S}{SYS}\n'
            f'Context: {context}\n'
            f'Question: {query}\n'
            f'Answer:{INST_E}'
        )
    else:
        prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

    resp = generate([prompt], max_new_tokens=MAX_GEN_TOKENS, num_beams=NUM_BEAMS)
    generated.append(clean_response(resp[0]))
    references.append(best_answer)

    if idx % 100 == 0:
        gc.collect(); torch.cuda.empty_cache()

elapsed = (time.time() - t0) / 60
print(f'\nDone in {elapsed:.1f} min')

Building FAISS index on ARC KB...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]


Running ARC Base (Standard RAG) on 1172 questions...


ARC-Base: 100%|██████████| 1172/1172 [37:36<00:00,  1.93s/it]


Done in 37.6 min


In [6]:
# ── Cell 6: Compute & Save Metrics ───────────────────────────────────────────
import pandas as pd

r1, r2, rl, ecs = compute_metrics(generated, references)
mauve_score     = compute_mauve(generated, references)
accuracy        = compute_accuracy(generated, arc)

result = {
    'method':   'Base-ARC',
    'dataset':  'ARC',
    'R1':    round(float(r1.mean()),  3),
    'R2':    round(float(r2.mean()),  3),
    'RL':    round(float(rl.mean()),  3),
    'ECS':   round(float(ecs.mean()), 3),
    'MAUVE': round(mauve_score,       3),
    'Accuracy': round(accuracy,       2),
}

print('\n' + '='*55)
print('ARC Base (Standard RAG, cosine-only) — Results')
print('='*55)
print(f'  R1     = {result["R1"]}')
print(f'  R2     = {result["R2"]}')
print(f'  RL     = {result["RL"]}')
print(f'  ECS    = {result["ECS"]}')
print(f'  MAUVE  = {result["MAUVE"]}')
print(f'  Acc    = {result["Accuracy"]}%')
print('='*55)

# Save
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_json = f'{OUTPUT_DIR}/arc_base_result_{ts}.json'
with open(out_json, 'w') as f:
    json.dump(result, f, indent=2)

df = pd.DataFrame({'generated': generated,
                   'reference': [r[0] if isinstance(r, list) else r for r in references]})
df.to_csv(f'{OUTPUT_DIR}/ARC_Base_{ts}.csv', index=False, quoting=1)

print(f'\nSaved → {out_json}')
print('\n*** Add this row to Table 1 in the paper: ***')
print(f'Base (ours, ARC): R1={result["R1"]}, ECS={result["ECS"]}, Acc={result["Accuracy"]}%')

Featurizing p:   0%|          | 0/1172 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/1172 [00:00<?, ?it/s]

WARNING clustering 2344 points to 117 centroids: please provide at least 4563 training points



ARC Base (Standard RAG, cosine-only) — Results
  R1     = 42.329
  R2     = 34.683
  RL     = 41.662
  ECS    = 56.177
  MAUVE  = 32.75
  Acc    = 58.79%

Saved → /home/user/RAG_Best_Practices/outputs/arc_base_result_20260430_085946.json

*** Add this row to Table 1 in the paper: ***
Base (ours, ARC): R1=42.329, ECS=56.177, Acc=58.79%
